# Get stats from `sacct` and parse them

This notebook does both the collecting of data from sacct (which can take a while) and the analysis of the data. Skip the data collection if the last run is reasonably complete so you can get to the analysis faster.

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

import re
import subprocess
from datetime import datetime, timedelta

In [ ]:
# Keep these values the same if you aren't getting new sacct data
START_DATE = datetime(2024,9,1)
END_DATE = datetime(2025,3,3)
filename = f'/shared/courseSharedFolders/135510outer/135510/sacctData/{START_DATE.strftime("%Y-%m-%d")}_to_{END_DATE.strftime("%Y-%m-%d")}.txt'

## Data collection

Skip this cell to not get new data

In [ ]:
output = subprocess.run(['sacct', '-XPa', '--delimiter', '`', '-o', 'ALL', '-S', START_DATE.strftime("%Y-%m-%d"), '-E', END_DATE.strftime("%Y-%m-%d")], capture_output=True)
with open(filename, 'wb') as fp:
    fp.write(output.stdout)
print("Done getting data from sacct")

## Data cleanup

There can be some funky issues with the data. This cleans them up.

### Fix newlines

There can be issues with newlines in the SubmitLine column. This section fixes those before loading the data so that the columns don't get messed up.

In [ ]:
def fix_newlines_in_rows(input_file_path, output_file_path, correct_backtick_count=2):
    """
    Function generated with GPT 4o

    The idea is to fix newlines in the SubmitLine column, which contains user input that isn't 
    restricted to a certain character set. It can use pipe characters, but I have yet to see 
    backticks in a submitline, so those are used as separators. This function checks to see if 
    there's the expected number of backticks in a row of the file, and if there isn't, it 
    appends the next line to the line that has the wrong number of backticks. Whatever it 
    modifies is printed out to the console.
    """
    with open(input_file_path, 'r') as file:
        lines = file.readlines()

    processed_lines = []
    i = 0
    while i < len(lines):
        line = lines[i].rstrip('\n')
        actual_backtick_count = line.count('`')
        
        # Check if the current line has the correct number of backticks
        if actual_backtick_count != correct_backtick_count:
            # If not, attempt to append the next line, if available
            if i + 1 < len(lines):
                line += ' ' + lines[i + 1].rstrip('\n')  # Appending the next line
                print(f"fixed line, now reads {line}\n")
                i += 1  # Skip the next line as it has been appended
            
        processed_lines.append(line)
        i += 1

    # Optionally write the processed lines to an output file
    with open(output_file_path, 'w') as file:
        for line in processed_lines:
            file.write(line + '\n')

In [ ]:
fixed_filename = filename.replace(".txt", "_fixed.txt")
fix_newlines_in_rows(filename, fixed_filename, correct_backtick_count=111)

### Load data into Pandas

In [ ]:
df = pd.read_csv(fixed_filename, sep="`")
len(df)

### Add columns

Add derived columns with start and end times as `datetime` objects, as well as time in hours on jobs, and cpu hours on jobs.

In [ ]:
def make_datetime(dt):
    if pd.isnull(dt):
        return np.nan
    elif dt == "Unknown":
        return np.nan
    else:
        try:
            return datetime.strptime(dt, "%Y-%m-%dT%H:%M:%S")
        except Exception as e:
            print(e)
            return np.nan

In [ ]:
# df['startDT'] = df.Start.apply(lambda x: datetime.strptime(x, "%Y-%m-%dT%H:%M:%S") if not pd.isnull(x) else np.nan)
df['startDT'] = df.Start.apply(make_datetime)

In [ ]:
# df['endDT'] = df.End.apply(lambda x: datetime.strptime(x, "%Y-%m-%dT%H:%M:%S") if not pd.isnull(x) or x != "Unknown" else np.nan)
df['endDT'] = df.End.apply(make_datetime)

In [ ]:
df['elapsedHours'] = df.ElapsedRaw.astype(float) / 3600
df.elapsedHours

In [ ]:
df['cpuHours'] = df.elapsedHours * df.ReqCPUS
df.cpuHours

### Node event time series

To track cluster capacity, this section creates a new `timeline` dataframe to hold info on events corresponding to the start and end of jobs, as well as the number of jobs running and nodes active in the cluster as a whole and the partition.

In [ ]:
def mixrange(s):
    """
    Taken from stackoverflow:
    https://stackoverflow.com/a/18759797
    """
    r = []
    for i in s.split(','):
        if '-' not in i:
            r.append(int(i))
        else:
            l,h = map(int, i.split('-'))
            r+= range(l,h+1)
    return r

In [ ]:
def expandNodeList(x):
    if re.match(r'.*\[[0-9,-]+\]', x):
        #do some stuff
        prefix, theRange = re.findall(r'(.*)\[([0-9,-]+)\]', x)[0]
        return [f'{prefix}{x}' for x in mixrange(theRange)]
    else:
        return [x]

In [ ]:
# Expand nodes into separate rows
exploded = df.assign(node=df['NodeList'].apply(expandNodeList)).explode('node').drop(columns='NodeList')

In [ ]:
# Create a sequence of events
node_usage = []

In [ ]:
for _, row in exploded.iterrows():
    start_event = {
        "type": "start",
        "dt": row['startDT'],
        "node": row["node"],
        "partition": row["Partition"],
        "jobId": row["JobID"]
    }
    end_event = start_event.copy()
    end_event.update({"type": "end", "dt": row['endDT']})
    node_usage.extend([start_event, end_event])

In [ ]:
timeline = pd.DataFrame(node_usage)

In [ ]:
timeline.sort_values('dt', inplace=True)
timeline

In [ ]:
nodeCounts = {}
for partition in timeline.partition.unique():
    nodeCounts[partition] = {}
    for node in timeline[timeline.partition == partition].node.unique():
        nodeCounts[partition][node] = 0

In [ ]:
timeline['nodeJobsRunning'] = 0
timeline['partitionJobsRunning'] = 0
timeline['totalJobsRunning'] = 0
timeline['nodesActive'] = 0
timeline['partitionNodesActive'] = 0
timeline.head()

**Side note: How the counts work**

The idea here is to iteratively use start and stop events to build counts for how many nodes are active at one time, and how many jobs are running on each node. There's a dict outside of the counting loop where counts are tracked as the timeline dataframe is iterated through, so that each event can get an accurate count. That dict is nested like `nodeCount[partition][node]`.

To get the number of jobs running in a partition, and to get the number of active nodes within a partition, I use some nested comprehensions to keep the calculation on one line. These are the proof of concept bits that showed me that I could make this work.

In [ ]:
test = {'general': {'g1': 2, 'g2': 0}, 'gpu': {'gpu1': 2, 'gpu2': 5}}

In [ ]:
sum(sum(v.values()) for v in test.values())

In [ ]:
sum(sum([x > 0 for x in v.values()]) for v in test.values())

In [ ]:
# This is where everything comes together to create the timeline of job events.
for i, row in timeline.iterrows():
    # For a start event, increment counts in dicts
    if row['type'] == 'start':
        nodeCounts[row['partition']][row['node']] += 1

    # For an end event, decrement counts in dicts
    elif row['type'] == 'end':
        nodeCounts[row['partition']][row['node']] -= 1

    # Get values to set in row
    nodeJobsRunning = nodeCounts[row['partition']][row['node']]
    partitionJobsRunning = sum(v for v in nodeCounts[row['partition']].values())
    partitionNodes = sum(v > 0 for v in nodeCounts[row['partition']].values())
    # Sum of all of the job counts for all nodes
    totalJobs = sum(sum(v.values()) for v in nodeCounts.values())
    # Count of all nodes with a job count greater than 0
    totalNodes = sum(sum([x > 0 for x in v.values()]) for v in nodeCounts.values())

    # Set the values in the row
    timeline.at[i, 'nodeJobsRunning'] = nodeJobsRunning
    timeline.at[i, 'partitionJobsRunning'] = partitionJobsRunning
    timeline.at[i, 'totalJobsRunning'] = totalJobs
    timeline.at[i, 'nodesActive'] = totalNodes
    timeline.at[i, 'partitionNodesActive'] = partitionNodes
timeline.head()

## Data analysis

Take the cleaned up data and analyze it.

Questions to answer:
- How many hours have nodes been running?
- What's the total usage?
- Who have been the top users, and how much have they been using the system?
  - Why is their job use high? Lots of jobs, long-running interactive jobs?
- Are we hitting a ceiling for node usage? (for each partition)

### Overall stats to report up

In [ ]:
print(f'Total users: {len(df.User.unique())}')
print(f'Total hours of use: {df.elapsedHours.sum():,.2f}')
print(f'Total CPU hours: {df.cpuHours.sum():,.2f} (job hours * CPUs per job)')

### Node occupancy

This gives an overview of whether node caps are being hit in the time frame under question, and provides an interface for generating a line graph showing node usage.

In [ ]:
partitionCaps = {
    'general': 75,
    'gpu': 75,
    'desktop': 50,
    'gpu-parallel': 20
}

In [ ]:
for partition, cap in partitionCaps.items():
    maximumActiveNodes = timeline[timeline.partition == partition].partitionNodesActive.max()
    print(f"In the `{partition}` queue, the maximum active nodes was {maximumActiveNodes}/{cap}")

In [ ]:
# General queue occupancy over time

start_dt = datetime(2025,1,1)
end_dt = datetime.now()
queue = 'general'

# Display line graph showing that activity
timeline[(timeline.dt >= start_dt) & (timeline.dt <= end_dt) & (timeline.partition == queue)].plot.line(x='dt', y='partitionNodesActive', figsize=(16,9))

In [ ]:
# GPU queue occupancy over time

start_dt = datetime(2025,1,1)
end_dt = datetime.now()
queue = 'gpu'

# Display line graph showing that activity
timeline[(timeline.dt >= start_dt) & (timeline.dt <= end_dt) & (timeline.partition == queue)].plot.line(x='dt', y='partitionNodesActive', figsize=(16,9))

### Usage by user

Check for out of the ordinary usage over a range of time. The first set of bar charts show the usage across the whole dataset collected from Slurm accounting.

In [ ]:
userUsage = df.pivot_table(values='cpuHours', index='User', columns='Partition', aggfunc='sum')

for partition in df.Partition.unique():
    partitionUsage = userUsage.sort_values(by=partition, ascending=False)
    ax = plt.subplot()
    plt.barh(y=partitionUsage.head(10).index, width=partitionUsage.head(10)[partition])
    plt.title(f"`{partition}` usage since {START_DATE.strftime('%B %-d, %Y')}")
    plt.xlabel('CPU Hours')
    plt.ylabel('Top 10 Users')
    plt.show()

#### Usage since start of term

This shows the usage since the start of term, assuming that the `startOfTerm` date is up to date.

In [ ]:
startOfTerm = datetime(2025,1,1)

timeBound = df[df.startDT >= startOfTerm]
userUsage = timeBound.pivot_table(values='cpuHours', index='User', columns='Partition', aggfunc='sum')

for partition in timeBound.Partition.unique():
    partitionUsage = userUsage.sort_values(by=partition, ascending=False)
    ax = plt.subplot()
    plt.barh(y=partitionUsage.head(10).index, width=partitionUsage.head(10)[partition])
    plt.title(f"`{partition}` usage since {startOfTerm.strftime('%B %-d, %Y')}")
    plt.xlabel('CPU Hours')
    plt.ylabel('Top 10 Users')
    plt.show()

#### Usage in past month

Technically, usage in the 30 days prior to the end date of the dataset

In [ ]:
oneMonthAgo = END_DATE - timedelta(days=30)

timeBound = df[df.startDT >= oneMonthAgo]
userUsage = timeBound.pivot_table(values='cpuHours', index='User', columns='Partition', aggfunc='sum')

for partition in timeBound.Partition.unique():
    partitionUsage = userUsage.sort_values(by=partition, ascending=False)
    ax = plt.subplot()
    plt.barh(y=partitionUsage.head(10).index, width=partitionUsage.head(10)[partition])
    plt.title(f"`{partition}` usage since {oneMonthAgo.strftime('%B %-d, %Y')}")
    plt.xlabel('CPU Hours')
    plt.ylabel('Top 10 Users')
    plt.show()

#### Usage in past week

Technically usage in the 7 days prior to the end date of the data collected

In [ ]:
oneWeekAgo = END_DATE - timedelta(days=7)

timeBound = df[df.startDT >= oneWeekAgo]
userUsage = timeBound.pivot_table(values='cpuHours', index='User', columns='Partition', aggfunc='sum')

for partition in timeBound.Partition.unique():
    partitionUsage = userUsage.sort_values(by=partition, ascending=False)
    ax = plt.subplot()
    plt.barh(y=partitionUsage.head(10).index, width=partitionUsage.head(10)[partition])
    plt.title(f"`{partition}` usage since {oneWeekAgo.strftime('%B %-d, %Y')}")
    plt.xlabel('CPU Hours')
    plt.ylabel('Top 10 Users')
    plt.show()